# 🧪 W2-D3 概念实验：GPT vs BERT 架构对比

> 配套阅读：`第2周-Day3-GPT与BERT架构对比.md`（完整原理、参数计算、业务关联在那边）
> 这个 notebook 用可执行实验回答三个问题：
> 1. **因果掩码如何限制 GPT 的注意力？** 为什么不能"看到未来"？
> 2. **参数量怎么算？** 为什么大模型时代 GPT 架构胜出？
> 3. **自回归生成的计算代价** — 每生成一个 token 都要重新计算什么？

## 实验 1：因果掩码 — GPT 的"单向注意力"

GPT 用因果掩码让每个 token 只能看到自己及之前的 token。
下面用 numpy 实现注意力，对比有/无掩码的效果。

In [ ]:
import numpy as np

np.random.seed(42)
seq_len = 5
d_k = 4

Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_k)

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

# 无掩码（BERT 双向注意力）
scores = Q @ K.T / np.sqrt(d_k)
attn_bert = softmax(scores, axis=-1)

# 因果掩码（GPT 单向注意力）
mask = np.triu(np.ones((seq_len, seq_len)), k=1) * (-1e9)
scores_masked = scores + mask
attn_gpt = softmax(scores_masked, axis=-1)

print("因果掩码（1=可见，0=被遮挡）:")
print((mask == 0).astype(int))
print()
print("BERT 注意力权重（每行=该 token 对所有位置的注意力）:")
print(np.round(attn_bert, 2))
print()
print("GPT 注意力权重（右上角应为 0）:")
print(np.round(attn_gpt, 2))
print()
print("验证：GPT 注意力上三角是否全为 0？")
upper_tri = attn_gpt[np.triu_indices(seq_len, k=1)]
print(f"  上三角最大值: {upper_tri.max():.2e} → {'✓ 单向注意力生效' if upper_tri.max() < 1e-6 else '✗ 失败'}")

## 实验 2：参数量计算 — 真实模型的参数是怎么分布的？

In [ ]:
import numpy as np

def calc_transformer_params(vocab, d_model, n_layers, n_heads, d_ff=None):
    """计算 Transformer 参数量（decoder-only GPT 风格）"""
    if d_ff is None:
        d_ff = 4 * d_model
    d_head = d_model // n_heads

    # Embedding
    embed = vocab * d_model
    # Position embedding (learned, max_seq=2048)
    pos_embed = 2048 * d_model
    # Per layer: attention (Q,K,V,O) + FFN (W1, W2) + 2 LayerNorm
    attn_per_layer = 4 * d_model * d_model  # Q,K,V,O projections
    ffn_per_layer = 2 * d_model * d_ff  # W1, W2
    ln_per_layer = 2 * 2 * d_model  # 2 LN, each has gamma + beta
    layer_total = attn_per_layer + ffn_per_layer + ln_per_layer
    # Final LN
    final_ln = 2 * d_model
    # Output head (tied or not)
    output_head = vocab * d_model  # can tie with embedding

    total = embed + pos_embed + n_layers * layer_total + final_ln + output_head
    return {
        'embedding': embed, 'position': pos_embed,
        'attn_total': n_layers * attn_per_layer,
        'ffn_total': n_layers * ffn_per_layer,
        'ln_total': n_layers * ln_per_layer + final_ln,
        'output': output_head,
        'total': total
    }

configs = {
    'GPT-2 Small':  {'vocab':50257, 'd_model':768,  'n_layers':12, 'n_heads':12},
    'GPT-2 Medium': {'vocab':50257, 'd_model':1024, 'n_layers':24, 'n_heads':16},
    'GPT-2 XL':    {'vocab':50257, 'd_model':1600, 'n_layers':48, 'n_heads':25},
    'GPT-3 175B':  {'vocab':50257, 'd_model':12288, 'n_layers':96, 'n_heads':96},
    'LLaMA-7B':    {'vocab':32000,  'd_model':4096, 'n_layers':32, 'n_heads':32},
}

print(f"{'模型':<14} {'Embed':>8} {'Attn':>8} {'FFN':>8} {'LN':>6} {'Total':>10}")
print("-" * 60)
for name, cfg in configs.items():
    p = calc_transformer_params(**cfg)
    m = p['total'] / 1e6
    print(f"{name:<14} {p['embedding']/1e6:>7.1f}M {p['attn_total']/1e6:>7.1f}M {p['ffn_total']/1e6:>7.1f}M {p['ln_total']/1e6:>5.1f}M {m:>8.1f}M")

print("\n结论：FFN 参数始终占大头（~2/3），Embedding 在大词表模型中也占比不小。")

## 实验 3：自回归生成的计算代价 — 为什么长文本生成那么慢？

GPT 每生成一个 token，需要对新 token 和所有之前 token 做注意力计算。
模拟这个过程，看看计算量随生成长度的增长。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

# 计算 FLOPs（简化版：只算 attention 的 Q@K^T 和 Attn@V）
# 生成第 t 个 token 时，需要计算 Q_t 与所有 K_1..K_t 的点积
seq_lens = np.arange(1, 513)
d_model = 4096
n_layers = 32
n_heads = 32
d_head = d_model // n_heads

# 每生成一个 token，attention 的 FLOPs = 2 * n_layers * seq_len * d_head（Q@K^T + Attn@V 简化）
# 累积到生成长度 T 的总 FLOPs
total_flops = []
cumulative = 0
for t in seq_lens:
    # 第 t 步：Q_t @ K_{1..t}^T 大小 t×d，Attn weights @ V_{1..t} 大小 d×t
    step_flops = 2 * n_layers * n_heads * (2 * t * d_head)
    cumulative += step_flops
    total_flops.append(cumulative)

total_flops = np.array(total_flops)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(seq_lens, total_flops / 1e9, '-', linewidth=1.5)
ax.set_xlabel('已生成 token 数')
ax.set_ylabel('累积 FLOPs (G)')
ax.set_title('自回归生成计算量随长度增长（累积 O(T²)）')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("关键洞察：")
print(f"- 生成 128 tokens 需要累积计算 {total_flops[127]/1e9:.1f} G FLOPs")
print(f"- 生成 512 tokens 需要累积计算 {total_flops[511]/1e9:.1f} G FLOPs")
print(f"- 4x 长度 → {total_flops[511]/total_flops[127]:.1f}x 计算量（接近 T² 增长）")
print("\n→ 这就是为什么需要 KV-Cache 和 Flash Attention（Day 5 主题）")